# Feature Engineering — Non-Temporal Approach

This notebook transforms event-level smartphone sensing data into an **instance-based tabular dataset**
suitable for standard (non-temporal) ML models.

Each row is one prediction instance: *patient i on date t*, using only information from
the `window_size` calendar days **before** `t`. The target is the mean mood on day `t`.

### Pipeline overview
1. **Daily aggregation** — collapse event records to one row per (patient, calendar day)
2. **Window feature engineering** — for each target day, aggregate the preceding window
3. **Train / val / test split** — time-wise per patient (no leakage)
4. **Window size comparison** — report dataset statistics for several candidate window sizes

In [20]:
import warnings
from pathlib import Path
from typing import Dict, List, Tuple
import os

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

In [21]:
DATA_DIR = Path(os.getcwd()).parent.parent / "data"
SOURCE_DATA_FILE_CLEANED = DATA_DIR / "1b_dataset_cleaned.parquet"

df = pd.read_parquet(SOURCE_DATA_FILE_CLEANED)
print(
    f"Loaded {len(df):,} records | "
    f"{df['id'].nunique()} patients | "
    f"{df['variable'].nunique()} variables"
)
print(f"Date range: {df['time'].min().date()} to {df['time'].max().date()}")
df.head()

Loaded 376,413 records | 27 patients | 19 variables
Date range: 2014-02-17 to 2014-06-09


,id,variable,time,value
0,AS14.01,activity,2014-03-20 22:00:00,0.071429
1,AS14.01,activity,2014-03-20 23:00:00,0.091667
2,AS14.01,activity,2014-03-21 00:00:00,0.008333
3,AS14.01,activity,2014-03-21 01:00:00,0.000000
4,AS14.01,activity,2014-03-21 02:00:00,0.000000


In [22]:
# ============================================================
# PIPELINE CONFIGURATION
# ============================================================

# Candidate history window sizes in calendar days.
# Window size is a hyperparameter; we compare on the validation set.
WINDOW_SIZES: List[int] = [1, 2, 3, 5, 6, 7, 14]
DEFAULT_WINDOW: int = 7

# Per-patient time-wise split fractions (must sum to 1).
# Splitting per-patient rather than by absolute date handles unequal
# observation periods across patients.
TRAIN_FRAC: float = 0.6
VAL_FRAC: float = 0.2
TEST_FRAC: float = 0.2

OUT_DIR = Path(os.getcwd()) / "output"
OUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# Variable grouping — determines the daily aggregation method.
# ------------------------------------------------------------------

# MEAN_VARS: continuous ratings or activity rates.
#   Multiple readings per day → daily mean is the right summary.
MEAN_VARS: List[str] = [
    "mood",
    "circumplex.arousal",
    "circumplex.valence",
    "activity",
]

# SUM_VARS: durations (seconds) or binary event flags.
#   call and sms values are always 1.0 in this dataset → sum == event count.
#   screen and appCat.* are session durations → total daily usage is informative.
SUM_VARS: List[str] = [
    "screen",
    "call",
    "sms",
    "appCat.builtin",
    "appCat.communication",
    "appCat.entertainment",
    "appCat.finance",
    "appCat.game",
    "appCat.office",
    "appCat.other",
    "appCat.social",
    "appCat.travel",
    "appCat.unknown",
    "appCat.utilities",
    "appCat.weather",
]

APP_CAT_VARS: List[str] = [v for v in SUM_VARS if v.startswith("appCat.")]

# Validate configured variables against the loaded data
known_vars = set(df["variable"].unique())
assert not (set(MEAN_VARS) - known_vars), (
    f"Unknown MEAN_VARS: {set(MEAN_VARS) - known_vars}"
)
assert not (set(SUM_VARS) - known_vars), (
    f"Unknown SUM_VARS: {set(SUM_VARS) - known_vars}"
)

print(
    f"Configuration OK | "
    f"{len(MEAN_VARS)} mean-vars, {len(SUM_VARS)} sum-vars | "
    f"Window sizes to compare: {WINDOW_SIZES}"
)

Configuration OK | 4 mean-vars, 15 sum-vars | Window sizes to compare: [1, 2, 3, 5, 6, 7, 14]


---
## Step 1: Daily Aggregation

Collapse event-level records to one row per **(patient, calendar day)**.

### Aggregation rules

| Variable group | Method | Rationale |
|---|---|---|
| `mood`, `circumplex.arousal/valence`, `activity` | **mean** | Ratings/rates; average of multiple daily readings gives the daily level |
| `screen`, `appCat.*` | **sum** | Session durations; total daily usage is the meaningful quantity |
| `call`, `sms` | **sum** | Values are always 1.0 in this dataset → sum = event count |

### Missingness preservation

For every variable we also produce:
- `{var}_n_obs` — number of raw event records on that day
- `{var}_observed` — binary flag (1 = at least one valid observation that day)

Days with **no records at all** for a variable remain as `NaN` in the primary column
and `0` in the `_observed` column. This keeps the signal of absence (e.g., no screen
usage on a day is different from low screen usage).

In [23]:
def aggregate_to_daily(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate event-level long-form data to patient-day wide format.

    Parameters
    ----------
    df : pd.DataFrame
        Long-form DataFrame with columns [id, variable, time, value].

    Returns
    -------
    pd.DataFrame
        Wide-form with one row per (id, date). Columns:
          id, date,
          {var}_mean   (MEAN_VARS),
          {var}_sum    (SUM_VARS),
          {var}_n_obs  (all variables),
          {var}_observed (binary, all variables).
        Variable names are sanitised: dots replaced by underscores.
    """
    df = df.copy()
    # Floor timestamps to midnight so all readings on the same calendar day
    # belong to the same group regardless of hour.
    df["date"] = df["time"].dt.normalize()

    grouped = df.groupby(["id", "date", "variable"])["value"]
    agg_mean = grouped.mean()  # used for MEAN_VARS
    agg_sum = grouped.sum()  # used for SUM_VARS
    agg_count = grouped.count()  # used for all: counts non-NaN values

    var_in_data = set(agg_count.index.get_level_values("variable"))

    frames: List[pd.Series] = []

    for var_orig in MEAN_VARS:
        if var_orig not in var_in_data:
            continue
        vs = var_orig.replace(".", "_")  # safe column name
        frames.append(agg_mean.xs(var_orig, level="variable").rename(f"{vs}_mean"))
        frames.append(agg_count.xs(var_orig, level="variable").rename(f"{vs}_n_obs"))

    for var_orig in SUM_VARS:
        if var_orig not in var_in_data:
            continue
        vs = var_orig.replace(".", "_")
        frames.append(agg_sum.xs(var_orig, level="variable").rename(f"{vs}_sum"))
        frames.append(agg_count.xs(var_orig, level="variable").rename(f"{vs}_n_obs"))

    # Combine: rows = all observed (id, date) combos; missing combos for a
    # given variable become NaN (outer join behaviour of pd.concat axis=1).
    daily_wide = pd.concat(frames, axis=1)
    daily_wide.index.names = ["id", "date"]
    daily_wide = daily_wide.reset_index()
    daily_wide.columns.name = None

    # Binary observed indicators
    for var_orig in MEAN_VARS + SUM_VARS:
        vs = var_orig.replace(".", "_")
        nobs_col = f"{vs}_n_obs"
        if nobs_col in daily_wide.columns:
            # fillna(0) so that NaN (never in data) maps to 0, not True
            daily_wide[f"{vs}_observed"] = (daily_wide[nobs_col].fillna(0) > 0).astype(
                np.int8
            )

    return daily_wide.sort_values(["id", "date"]).reset_index(drop=True)

In [24]:
daily_wide = aggregate_to_daily(df)

n_patients = daily_wide["id"].nunique()
n_patient_days = len(daily_wide)
mood_obs_per_patient = daily_wide.groupby("id")["mood_observed"].sum()

print("Daily aggregation complete.")
print(f"  Shape            : {daily_wide.shape}")
print(f"  Patient-day rows : {n_patient_days:,}")
print(f"  Patients         : {n_patients}")
print(
    f"  Date range       : {daily_wide['date'].min().date()} to {daily_wide['date'].max().date()}"
)
print(
    f"  Mood obs / patient (min/median/max): "
    f"{mood_obs_per_patient.min():.0f} / "
    f"{mood_obs_per_patient.median():.0f} / "
    f"{mood_obs_per_patient.max():.0f}"
)

print("\nColumn overview (first 12):")
print(daily_wide.columns[:12].tolist())
daily_wide.head(3)

Daily aggregation complete.
  Shape            : (1973, 59)
  Patient-day rows : 1,973
  Patients         : 27
  Date range       : 2014-02-17 to 2014-06-09
  Mood obs / patient (min/median/max): 30 / 46 / 68

Column overview (first 12):
['id', 'date', 'mood_mean', 'mood_n_obs', 'circumplex_arousal_mean', 'circumplex_arousal_n_obs', 'circumplex_valence_mean', 'circumplex_valence_n_obs', 'activity_mean', 'activity_n_obs', 'screen_sum', 'screen_n_obs']


,id,date,mood_mean,mood_n_obs,circumplex_arousal_mean,circumplex_arousal_n_obs,circumplex_valence_mean,circumplex_valence_n_obs,activity_mean,activity_n_obs,...,appCat_entertainment_observed,appCat_finance_observed,appCat_game_observed,appCat_office_observed,appCat_other_observed,appCat_social_observed,appCat_travel_observed,appCat_unknown_observed,appCat_utilities_observed,appCat_weather_observed
0,AS14.01,2014-02-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
1,AS14.01,2014-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
2,AS14.01,2014-02-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


---
## Steps 2 & 3: Target Construction and Window Feature Engineering

### Target
For each **(patient, target_date)** instance:
- `target_mood` = mean of all mood readings on `target_date`
- Only days where mood was observed become target dates.

### Leakage prevention (by construction)
The history window for target date `t` spans **`[t - window_size, t - 1]`** inclusive.
No data from day `t` is ever included in features.
Target dates without at least `window_size` days of history are dropped.

### Feature families

| Family | Variables | Notes |
|---|---|---|
| Rolling mean / std / min / max | mood, arousal, valence, activity | Typical level and variability over window |
| Rolling sum / mean / max | screen, call, sms, appCat.* | Behavioral volume |
| Lag features | mood: full set up to `w`; others: lag-1 only | Recency; mood yesterday is very predictive |
| Same-weekday lag | mood | Weekly periodicity |
| Linear trend (slope) | mood (≥3 observed days) | Direction of change |
| Missingness indicators | all variables | Absence of signal may be informative |
| App category proportions | appCat.* | Usage composition, independent of volume |
| Temporal context | target date | Cyclical day-of-week, weekend flag |
| Window coverage | all | How much of the window had any data |

In [25]:
def _compute_window_features(
    window: pd.DataFrame,
    patient_daily: pd.DataFrame,
    target_date: pd.Timestamp,
    window_size: int,
) -> Dict:
    """
    Compute all features for a single (patient, target_date) instance.

    Parameters
    ----------
    window : pd.DataFrame
        Slice of patient_daily for the window [target_date-window_size, target_date-1].
        Indexed by Timestamp. NaN rows = calendar days with no observations.
        Length is always exactly window_size (guaranteed by caller).
    patient_daily : pd.DataFrame
        Full patient data indexed by date. Used for lag lookups that may fall
        outside the current window (e.g. same-weekday lag).
    target_date : pd.Timestamp
        Prediction target day. NOT included in window.
    window_size : int
        Number of calendar days in the window.
    """
    feats: Dict = {}

    # ------------------------------------------------------------------
    # Temporal context of the TARGET day
    # ------------------------------------------------------------------
    dow = target_date.dayofweek  # 0 = Monday, 6 = Sunday
    feats["target_dow"] = dow
    feats["target_is_weekend"] = int(dow >= 5)
    # Cyclical encoding so Monday (0) and Sunday (6) are close numerically
    feats["target_dow_sin"] = float(np.sin(2 * np.pi * dow / 7))
    feats["target_dow_cos"] = float(np.cos(2 * np.pi * dow / 7))

    # ------------------------------------------------------------------
    # Overall window coverage
    # A day counts as "observed" if any variable has a non-NaN value.
    # ------------------------------------------------------------------
    any_obs = window.notna().any(axis=1)
    feats["window_n_any_obs_days"] = int(any_obs.sum())
    feats["window_coverage_frac"] = float(any_obs.mean())

    # ------------------------------------------------------------------
    # MEAN variables: mood, circumplex.arousal/valence, activity
    # ------------------------------------------------------------------
    for var_orig in MEAN_VARS:
        vs = var_orig.replace(".", "_")
        mean_col = f"{vs}_mean"
        obs_col = f"{vs}_observed"

        if mean_col not in window.columns:
            continue

        series = window[mean_col]  # NaN = not observed that day
        obs = (
            window[obs_col].fillna(0)
            if obs_col in window.columns
            else series.notna().astype(float)
        )
        n_obs = int(obs.sum())

        # Rolling statistics (pandas skips NaN by default for mean/std/etc.)
        feats[f"{vs}_win_mean"] = float(series.mean()) if n_obs > 0 else np.nan
        feats[f"{vs}_win_std"] = float(series.std()) if n_obs > 1 else np.nan
        feats[f"{vs}_win_min"] = float(series.min()) if n_obs > 0 else np.nan
        feats[f"{vs}_win_max"] = float(series.max()) if n_obs > 0 else np.nan

        # Missingness indicators
        feats[f"{vs}_n_obs_days"] = n_obs
        feats[f"{vs}_obs_frac"] = n_obs / window_size
        feats[f"{vs}_n_missing_days"] = window_size - n_obs

        # Lag features
        # mood: full set up to window_size (each lag is an independent feature)
        # other MEAN_VARS: only lag_1 (most recent value — reduces feature count)
        max_lag = window_size if var_orig == "mood" else 1
        for lag in range(1, max_lag + 1):
            lag_date = target_date - pd.Timedelta(days=lag)
            if lag_date in patient_daily.index and mean_col in patient_daily.columns:
                feats[f"{vs}_lag{lag}"] = patient_daily.at[lag_date, mean_col]
            else:
                feats[f"{vs}_lag{lag}"] = np.nan

        # Same-weekday lag (7 days back) — captures weekly mood periodicity.
        # Added for all MEAN_VARS, but most useful for mood.
        swd_date = target_date - pd.Timedelta(days=7)
        if swd_date in patient_daily.index and mean_col in patient_daily.columns:
            feats[f"{vs}_same_wday_lag7"] = patient_daily.at[swd_date, mean_col]
        else:
            feats[f"{vs}_same_wday_lag7"] = np.nan

        # Linear trend over observed window values (mood only, need >= 3 points).
        # Positive slope = mood improving over the window.
        if var_orig == "mood":
            valid = series.dropna()
            if len(valid) >= 3:
                x = np.arange(len(valid), dtype=float)
                slope, *_ = stats.linregress(x, valid.values)
                feats[f"{vs}_trend"] = float(slope)
            else:
                feats[f"{vs}_trend"] = np.nan

    # ------------------------------------------------------------------
    # SUM variables: screen, call, sms, appCat.*
    # ------------------------------------------------------------------
    for var_orig in SUM_VARS:
        vs = var_orig.replace(".", "_")
        sum_col = f"{vs}_sum"
        obs_col = f"{vs}_observed"

        if sum_col not in window.columns:
            continue

        obs = (
            window[obs_col].fillna(0)
            if obs_col in window.columns
            else window[sum_col].notna().astype(float)
        )
        n_obs = int(obs.sum())

        # NaN (not observed) is treated as 0 for volume features.
        # This is reasonable: a day without a screen record likely means no usage,
        # not a missing sensor reading. The _obs_frac feature captures the distinction.
        series_filled = window[sum_col].fillna(0)
        series_raw = window[sum_col]  # keep NaN for max (avoid inflating with 0)

        feats[f"{vs}_win_sum"] = float(series_filled.sum())
        feats[f"{vs}_win_mean"] = float(series_filled.mean())  # includes 0-days
        feats[f"{vs}_win_max"] = float(series_raw.max()) if n_obs > 0 else np.nan
        feats[f"{vs}_n_obs_days"] = n_obs
        feats[f"{vs}_obs_frac"] = n_obs / window_size

    # ------------------------------------------------------------------
    # App category proportions
    # Share of total app usage per category; independent of total volume.
    # Denominator is sum across all appCat variables in the window.
    # ------------------------------------------------------------------
    total_app = sum(
        window[f"{v.replace('.', '_')}_sum"].fillna(0).sum()
        for v in APP_CAT_VARS
        if f"{v.replace('.', '_')}_sum" in window.columns
    )
    for var_orig in APP_CAT_VARS:
        vs = var_orig.replace(".", "_")
        sum_col = f"{vs}_sum"
        if sum_col in window.columns:
            var_total = window[sum_col].fillna(0).sum()
            feats[f"{vs}_prop"] = (var_total / total_app) if total_app > 0 else np.nan
        else:
            feats[f"{vs}_prop"] = np.nan

    return feats


def build_feature_matrix(
    daily_wide: pd.DataFrame,
    window_size: int,
) -> pd.DataFrame:
    """
    Build an instance-based feature matrix.

    For every (patient, target_date) pair where mood was observed on target_date,
    compute window features from the preceding `window_size` calendar days.
    Instances where not enough history is available are excluded.

    Parameters
    ----------
    daily_wide : pd.DataFrame
        Output of aggregate_to_daily().
    window_size : int
        Number of calendar days of history to use (>= 1).

    Returns
    -------
    pd.DataFrame
        One row per (id, target_date). First columns: id, target_date, target_mood.
        Remaining columns: engineered features.
    """
    if window_size < 1:
        raise ValueError(f"window_size must be >= 1, got {window_size}")

    records = []

    for patient_id, patient_df in daily_wide.groupby("id"):
        patient_df = patient_df.drop(columns="id").sort_values("date").set_index("date")

        # Expand to full calendar range so missing days have explicit NaN rows.
        # This ensures a window of size w always spans exactly w calendar days,
        # and missingness is captured structurally rather than by absence of rows.
        full_range = pd.date_range(
            patient_df.index.min(),
            patient_df.index.max(),
            freq="D",
        )
        patient_df = patient_df.reindex(full_range)

        mood_col = "mood_mean"
        if mood_col not in patient_df.columns:
            continue

        # Only predict for days where the target is available
        target_dates = patient_df.index[patient_df[mood_col].notna()]
        first_date = patient_df.index.min()

        for t in target_dates:
            window_start = t - pd.Timedelta(days=window_size)
            window_end = t - pd.Timedelta(days=1)

            # Require the full window to lie within the patient's observation period
            if window_start < first_date:
                continue

            window = patient_df.loc[window_start:window_end]

            # After reindexing, the window should always have exactly window_size rows.
            # Guard against unexpected edge cases without raising hard errors.
            if len(window) != window_size:
                continue

            feats = _compute_window_features(
                window=window,
                patient_daily=patient_df,
                target_date=t,
                window_size=window_size,
            )
            feats["id"] = patient_id
            feats["target_date"] = t
            feats["target_mood"] = patient_df.at[t, mood_col]
            records.append(feats)

    if not records:
        raise ValueError(
            f"No valid instances for window_size={window_size}. "
            "The window may be larger than any patient's observation period."
        )

    result = pd.DataFrame(records)

    # --- Post-construction integrity checks ---
    dups = result.duplicated(subset=["id", "target_date"])
    if dups.any():
        raise ValueError(f"{dups.sum()} duplicate (id, target_date) pairs found.")

    if result["target_mood"].isna().any():
        raise ValueError("Some target_mood values are NaN — unexpected.")

    # Move metadata columns to the front for readability
    meta_cols = ["id", "target_date", "target_mood"]
    feat_cols = [c for c in result.columns if c not in meta_cols]
    return result[meta_cols + feat_cols].reset_index(drop=True)

In [26]:
print(f"Building feature matrix for default window_size={DEFAULT_WINDOW}...")
feature_df = build_feature_matrix(daily_wide, window_size=DEFAULT_WINDOW)

meta_cols = ["id", "target_date", "target_mood"]
feat_cols = [c for c in feature_df.columns if c not in meta_cols]

print(f"\nFeature matrix — window_size={DEFAULT_WINDOW}")
print(f"  Total instances : {len(feature_df):,}")
print(f"  Feature columns : {len(feat_cols)}")
print(f"  Patients        : {feature_df['id'].nunique()}")
print(
    f"  Date range      : {feature_df['target_date'].min().date()} to {feature_df['target_date'].max().date()}"
)
print("\nInstances per patient:")
print(feature_df.groupby("id").size().describe().to_string())
print("\nTarget mood statistics:")
print(feature_df["target_mood"].describe().to_string())
print(
    f"\nOverall feature missingness : {feature_df[feat_cols].isna().mean().mean():.1%}"
)

Building feature matrix for default window_size=7...

Feature matrix — window_size=7
  Total instances : 1,261
  Feature columns : 136
  Patients        : 27
  Date range      : 2014-02-26 to 2014-06-08

Instances per patient:
count    27.000000
mean     46.703704
std       8.018319
min      30.000000
25%      42.000000
50%      46.000000
75%      53.000000
max      68.000000

Target mood statistics:
count    1261.000000
mean        6.990675
std         0.735120
min         3.000000
25%         6.600000
50%         7.000000
75%         7.500000
max         9.333333

Overall feature missingness : 5.5%


In [27]:
# Feature missingness breakdown by family
families = {
    "mood_lags": [c for c in feat_cols if "mood_lag" in c],
    "mood_rolling": [c for c in feat_cols if c.startswith("mood_win")],
    "mood_misc": [
        c
        for c in feat_cols
        if c.startswith("mood_") and "lag" not in c and "win" not in c
    ],
    "arousal": [c for c in feat_cols if "arousal" in c],
    "valence": [c for c in feat_cols if "valence" in c],
    "activity": [c for c in feat_cols if c.startswith("activity_")],
    "screen": [c for c in feat_cols if "screen" in c],
    "call_sms": [c for c in feat_cols if "call" in c or "sms" in c],
    "appCat": [c for c in feat_cols if "appCat" in c and "prop" not in c],
    "appCat_prop": [c for c in feat_cols if "appCat" in c and "prop" in c],
    "temporal": [c for c in feat_cols if "target_" in c],
    "window_cov": [c for c in feat_cols if c.startswith("window_")],
}

summary_rows = []
for family, cols in families.items():
    if cols:
        miss = feature_df[cols].isna().mean().mean()
        summary_rows.append(
            {"family": family, "n_features": len(cols), "missingness": f"{miss:.1%}"}
        )

pd.DataFrame(summary_rows).set_index("family")

,n_features,missingness
family,,
mood_lags,7,10.1%
mood_rolling,4,2.7%
mood_misc,4,1.6%
arousal,9,3.5%
valence,9,3.5%
activity,9,7.1%
screen,5,1.1%
call_sms,10,1.5%
appCat,60,7.2%


---
## Step 4: Train / Validation / Test Split

We split **per patient** in time order (no random shuffling).

For each patient:
- Sort instances by `target_date` (ascending)
- First 60% → **train**
- Next 20%  → **validation**
- Last 20%  → **test**

### Why per-patient rather than by absolute calendar date?
Patients have different observation durations and start dates.
A global date cutoff would leave some patients with very few training instances
and others with none in val/test. Per-patient relative splits give every patient
a proportional contribution to each split.

### Leakage check
After splitting, we verify that no `target_date` appears in two splits for the same patient.

In [28]:
def time_split(
    feature_df: pd.DataFrame,
    train_frac: float = TRAIN_FRAC,
    val_frac: float = VAL_FRAC,
    test_frac: float = TEST_FRAC,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split feature_df into train / val / test using per-patient time ordering.

    For each patient, instances are sorted by target_date and divided at
    fractional positions. No date from a later split can appear in an earlier
    split (within a patient).

    Parameters
    ----------
    feature_df : pd.DataFrame
        Output of build_feature_matrix().
    train_frac, val_frac, test_frac : float
        Fractions summing to 1.0.

    Returns
    -------
    Tuple of (train, val, test) DataFrames.
    """
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6, (
        "train_frac + val_frac + test_frac must equal 1.0"
    )

    train_parts, val_parts, test_parts = [], [], []

    for patient_id, pat_df in feature_df.groupby("id"):
        pat_df = pat_df.sort_values("target_date").reset_index(drop=True)
        n = len(pat_df)

        train_end = int(np.floor(n * train_frac))
        val_end = int(np.floor(n * (train_frac + val_frac)))

        train_parts.append(pat_df.iloc[:train_end])
        val_parts.append(pat_df.iloc[train_end:val_end])
        test_parts.append(pat_df.iloc[val_end:])

    train = pd.concat(train_parts, ignore_index=True)
    val = pd.concat(val_parts, ignore_index=True)
    test = pd.concat(test_parts, ignore_index=True)

    # Integrity check: no date overlap within a patient across splits
    for pid in feature_df["id"].unique():
        tr = set(train.loc[train["id"] == pid, "target_date"])
        va = set(val.loc[val["id"] == pid, "target_date"])
        te = set(test.loc[test["id"] == pid, "target_date"])
        for s1, s2, label in [
            (tr, va, "train/val"),
            (va, te, "val/test"),
            (tr, te, "train/test"),
        ]:
            overlap = s1 & s2
            if overlap:
                raise ValueError(
                    f"Patient {pid}: date overlap in {label}: {sorted(overlap)}"
                )

    return train, val, test

In [29]:
train, val, test = time_split(feature_df)

print(f"Split sizes (window_size={DEFAULT_WINDOW}):")
print(f"  Train : {len(train):>4} instances ({len(train) / len(feature_df):.0%})")
print(f"  Val   : {len(val):>4} instances ({len(val) / len(feature_df):.0%})")
print(f"  Test  : {len(test):>4} instances ({len(test) / len(feature_df):.0%})")

print("\nDate ranges per split:")
for name, split in [("Train", train), ("Val", val), ("Test", test)]:
    print(
        f"  {name:5s}: {split['target_date'].min().date()} to {split['target_date'].max().date()}"
    )

print("\nTarget mood mean by split:")
for name, split in [("Train", train), ("Val", val), ("Test", test)]:
    print(
        f"  {name:5s}: {split['target_mood'].mean():.3f} ± {split['target_mood'].std():.3f}"
    )

Split sizes (window_size=7):
  Train :  744 instances (59%)
  Val   :  252 instances (20%)
  Test  :  265 instances (21%)

Date ranges per split:
  Train: 2014-02-26 to 2014-05-17
  Val  : 2014-04-09 to 2014-05-28
  Test : 2014-04-17 to 2014-06-08

Target mood mean by split:
  Train: 7.004 ± 0.744
  Val  : 6.922 ± 0.699
  Test : 7.018 ± 0.743


---
## Step 5: Window Size Comparison

Compare candidate window sizes `[1, 2, 3, 5, 7, 14]` on the validation split.

For each window size we report:
- **Instance counts** per split (larger windows exclude early dates → fewer instances)
- **Feature count** (grows with window_size due to additional lag features)
- **Val missingness** — fraction of feature values that are NaN in val set
- **Val baseline MAE** — mean absolute error of predicting the training set mean
  (a trivial baseline; lower = easier task)

In [30]:
def compare_window_sizes(
    daily_wide: pd.DataFrame,
    window_sizes: List[int] = WINDOW_SIZES,
    train_frac: float = TRAIN_FRAC,
    val_frac: float = VAL_FRAC,
    test_frac: float = TEST_FRAC,
) -> pd.DataFrame:
    """
    Build feature matrices for multiple window sizes and report dataset statistics.

    Parameters
    ----------
    daily_wide : pd.DataFrame
        Output of aggregate_to_daily().
    window_sizes : List[int]
        Candidate window sizes to evaluate.

    Returns
    -------
    pd.DataFrame
        Summary table with one row per window size.
    """
    rows = []

    for ws in window_sizes:
        print(f"Window {ws:>2}d ... ", end="", flush=True)

        feat_df = build_feature_matrix(daily_wide, window_size=ws)
        tr, va, te = time_split(feat_df, train_frac, val_frac, test_frac)

        meta_cols = ["id", "target_date", "target_mood"]
        feat_cols = [c for c in feat_df.columns if c not in meta_cols]

        val_missing = float(va[feat_cols].isna().mean().mean())
        baseline_mae = float(
            (va["target_mood"] - tr["target_mood"].mean()).abs().mean()
        )

        rows.append(
            {
                "window_size": ws,
                "n_train": len(tr),
                "n_val": len(va),
                "n_test": len(te),
                "n_total": len(feat_df),
                "n_features": len(feat_cols),
                "val_missing_frac": round(val_missing, 3),
                "val_baseline_mae": round(baseline_mae, 4),
            }
        )

        print(
            f"total={len(feat_df):>4} | "
            f"train={len(tr):>3} val={len(va):>3} test={len(te):>3} | "
            f"feats={len(feat_cols):>3} | "
            f"missing={val_missing:.1%} | "
            f"MAE={baseline_mae:.4f}"
        )

    return pd.DataFrame(rows)

In [31]:
comparison = compare_window_sizes(daily_wide)
print("\n--- Summary table ---")
comparison

Window  1d ... total=1268 | train=749 val=253 test=266 | feats=130 | missing=10.0% | MAE=0.5035
Window  2d ... total=1268 | train=749 val=253 test=266 | feats=131 | missing=5.5% | MAE=0.5035
Window  3d ... total=1267 | train=749 val=252 test=266 | feats=132 | missing=4.1% | MAE=0.4961
Window  5d ... total=1265 | train=748 val=252 test=265 | feats=134 | missing=3.4% | MAE=0.4895
Window  6d ... total=1263 | train=746 val=252 test=265 | feats=135 | missing=3.3% | MAE=0.4895
Window  7d ... total=1261 | train=744 val=252 test=265 | feats=136 | missing=3.1% | MAE=0.4892
Window 14d ... total=1245 | train=735 val=248 test=262 | feats=143 | missing=2.5% | MAE=0.4760

--- Summary table ---


,window_size,n_train,n_val,n_test,n_total,n_features,val_missing_frac,val_baseline_mae
0,1,749,253,266,1268,130,0.100,0.5035
1,2,749,253,266,1268,131,0.055,0.5035
2,3,749,252,266,1267,132,0.041,0.4961
3,5,748,252,265,1265,134,0.034,0.4895
4,6,746,252,265,1263,135,0.033,0.4895
5,7,744,252,265,1261,136,0.031,0.4892
6,14,735,248,262,1245,143,0.025,0.4760


In [32]:
# Save datasets for the default window size
print(f"Saving datasets for window_size={DEFAULT_WINDOW} to {OUT_DIR}/...")

daily_wide.to_parquet(OUT_DIR / "daily_wide.parquet", index=False)
feature_df.to_parquet(OUT_DIR / f"features_w{DEFAULT_WINDOW}.parquet", index=False)
train.to_parquet(OUT_DIR / f"train_w{DEFAULT_WINDOW}.parquet", index=False)
val.to_parquet(OUT_DIR / f"val_w{DEFAULT_WINDOW}.parquet", index=False)
test.to_parquet(OUT_DIR / f"test_w{DEFAULT_WINDOW}.parquet", index=False)

print("Saved:")
for p in sorted(OUT_DIR.glob("*.parquet")):
    print(f"  {p.name}")

Saving datasets for window_size=7 to /Users/jortverbeek/Desktop/VU/data mining/Assignment 1/data_mining_group_9_assignment_1/src/1c_feature_engineering_non_temporal/output/...
Saved:
  daily_wide.parquet
  features_w7.parquet
  test_w7.parquet
  train_w7.parquet
  val_w7.parquet


---
## Summary of design decisions

### Aggregation
- `mood`, `arousal`, `valence`, `activity` → **daily mean**. These are subjective ratings
  or normalized rates; averaging multiple readings gives the representative daily value.
- `screen`, `appCat.*` → **daily sum**. These are session durations; total daily usage is the
  natural summary.
- `call`, `sms` → **daily sum** (= event count, since each record has value 1.0).

### Missingness
- Days with no records for a variable contribute `NaN` to the primary column **and** `0` to
  the `_observed` indicator. This keeps the absence signal: if a patient had no screen
  usage on a day, that is behaviorally meaningful and should not be silently imputed.
- For SUM variables in window features, unobserved days are treated as 0 usage for
  volume aggregates (win_sum, win_mean), but the `_n_obs_days` / `_obs_frac` features
  separately capture how many days were actually observed.
- No imputation is done inside the feature engineering step.

### Feature design
- **Full lag set for mood** (lag₁ … lagₙ) because yesterday's mood is empirically the
  strongest predictor. For other MEAN_VARS we only include lag₁ to limit feature dimensionality.
- **Same-weekday lag (lag₇)** captures weekly rhythms without requiring a 7-day window.
- **Linear slope** of mood over the window captures whether mood is trending up or down.
- **App proportion** features normalize app usage by total app time, capturing behavioral
  composition independently of total phone usage volume.
- **Cyclical day-of-week encoding** (sin/cos) ensures Monday and Sunday are numerically
  close.

### Split strategy
- Per-patient relative time splits (60/20/20 by sorted date index).
- Avoids leakage: all features for instance `(patient, t)` use only days `< t`.
- Rationale for per-patient vs global split: patients have different observation periods;
  a global date cutoff would give some patients near-zero val/test instances.

### Open questions / next steps
- Should mood imputation from the cleaning step be re-visited before daily aggregation?
- Should we include inter-patient features (e.g. population mean on the same day)?
- After choosing the best window size on val, re-fit on train+val before evaluating on test.
- Consider patient-level normalization (subtract per-patient mood mean) to focus the model
  on within-patient variation rather than between-patient baseline differences.